In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Corte de fios para estimativa de valores esperados

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimativa de uso: 22 segundos em um processador Heron (NOTA: Isso é apenas uma estimativa. Seu tempo de execução pode variar.)*
## Resultados de aprendizagem
Após concluir este tutorial, os usuários devem entender:
- Como usar o [`qiskit-addon-cutting`](https://github.com/Qiskit/qiskit-addon-cutting) para particionar um circuito grande em subcircuitos menores, reduzindo assim o efeito do ruído

## Pré-requisitos
Sugerimos que os usuários estejam familiarizados com o seguinte tópico antes de prosseguir com este tutorial:
- Usar a primitiva [Sampler](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), que é utilizada neste fluxo de trabalho

## Contexto
Circuit-knitting é um termo abrangente que engloba vários métodos de particionamento de um circuito em múltiplos subcircuitos menores envolvendo menos portas ou qubits. Cada um dos subcircuitos pode ser executado independentemente, e o resultado final é obtido por algum pós-processamento clássico sobre o resultado de cada subcircuito. Esta técnica está acessível no [addon Qiskit de corte de circuito](https://qiskit.github.io/qiskit-addon-cutting/index.html); consulte a [documentação](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) junto com outro [material introdutório](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html) para uma explicação detalhada da técnica.

Este tutorial foca em um método chamado **corte de fios**, onde o circuito é particionado ao longo do fio [\[1\], \[2\]](#references). Observe que o particionamento é simples em circuitos clássicos, pois o resultado no ponto de partição pode ser determinado deterministicamente, e é 0 ou 1. No entanto, o estado do qubit no ponto do corte é, em geral, um estado misto. Portanto, cada subcircuito precisa ser medido múltiplas vezes em diferentes bases (geralmente uma base tomograficamente completa, como a base de Pauli [\[3\], \[4\]](#references)) e correspondentemente preparado em seu autoestado. A figura abaixo (cortesia: [\[7\]](#references)) mostra um exemplo de corte de fios para um estado GHZ de quatro qubits em três subcircuitos. Aqui $M_j$ denota um conjunto de bases (geralmente Pauli X, Y e Z), e $P_i$ denota um conjunto de autoestados (geralmente $|0\rangle$, $|1\rangle$, $|+\rangle$ e $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Como cada subcircuito tem menos qubits e portas, espera-se que eles sejam menos suscetíveis ao ruído. Este tutorial mostra um exemplo onde este método pode ser usado para suprimir efetivamente o ruído no sistema.
## Requisitos
Antes de iniciar este tutorial, certifique-se de ter o seguinte instalado:

- Qiskit SDK v2.0 ou posterior, com suporte de [visualização](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 ou posterior ( `pip install qiskit-ibm-runtime` )
- Addon Qiskit de corte de circuito v0.10.0 ou posterior (`pip install qiskit-addon-cutting`)
- Qiskit addon utils 0.3 ou posterior (`pip install qiskit-addon-utils`)
- Qiskit Aer (`pip install qiskit-aer` )
## Configuração

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Exemplo em simulador de pequena escala
Este tutorial implementa um [padrão Qiskit](/guides/intro-to-patterns) para simular um circuito de Localização de Muitos Corpos (MBL) unidimensional (1D). O circuito MBL é um circuito eficiente em hardware e é parametrizado por dois parâmetros $\theta$ e $\vec{\phi}$. Quando $\theta$ é definido como $0$ e o estado inicial é preparado em $|0\rangle$ para todos os qubits, o valor esperado ideal de $\langle Z_i \rangle$ é $+1$ para cada site de qubit $i$ independentemente dos valores de $\vec{\phi}$. Mais detalhes sobre este circuito estão disponíveis neste [artigo](https://www.nature.com/articles/s41467-025-57623-x).

Observe que em um simulador sem ruído, o valor esperado obtido com e sem corte de circuito será o mesmo.
### Passo 1: Mapear entradas clássicas para um problema quântico
#### Construir o circuito MBL 1D
Primeiro, apresentamos uma função para construir o circuito MBL 1D.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Calculamos o valor esperado médio $O = \frac{1}{n} \sum_i Z_i$ sobre todos os qubits para $\theta = 0$. Como o valor esperado ideal de $\langle Z_i \rangle = 1$ $\forall$ $i$, o valor esperado ideal de $O$ também é $1$. Os parâmetros $\phi$ são selecionados aleatoriamente.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

O circuito precisa ser anotado inserindo **CutWire** nos locais desejados para particioná-lo. Para este tutorial, optamos por uma partição igual. O circuito MBL é projetado de forma que definir `use_cut=True` na função insira a anotação adequadamente após $\frac{n}{2}$ qubits, sendo $n$ o número de qubits no circuito original. Também atribuímos os parâmetros gerados aleatoriamente ao circuito.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif)

### Passo 2: Otimizar problema para execução em hardware quântico
#### Cortar o circuito em subcircuitos menores
Agora particionamos o circuito em dois subcircuitos menores usando [`qiskit-addon-cutting`](https://qiskit.github.io/qiskit-addon-cutting/). `qiskit-addon-cutting` acrescenta um gate virtual `Move` para dividir o local de corte de fio ajustando adequadamente o número de qubits. Agora criamos o circuito com este gate virtual. Como há um corte de fio, o número de qubits associados aumentará em 1.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif)

#### Construir e expandir o observável
O observável, conforme definido anteriormente, será a média de $Z$ em cada qubit. No entanto, ao inserir o gate virtual `Move`, o número efetivo de qubits no circuito aumenta. O observável também deve ser expandido de acordo para contabilizar essa mudança no número de qubits. Observe que o observável sempre age de forma trivial (como $I$) no qubit extra adicionado para o gate virtual `Move`.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Now the circuit can be partitioned along the `Move` gate and we obtain the subcircuits, as well as the subobservable, which is the portion of the original observable associated with each subcircuit.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Aqui visualizamos os dois subcircuitos:

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif)

Expandir o observável usando a operação `Move` requer uma estrutura de dados `PauliList`. Para reconstruir o valor esperado do circuito original, precisamos do observável no formato `SparsePauliOp`.

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


Conforme discutido anteriormente, para cada corte o circuito upstream deve ser medido em uma base de Pauli, e o circuito downstream deve ser preparado no autoestado da base. A função `generate_cutting_experiments` cria todos esses circuitos necessários e os coeficientes associados a cada circuito necessários para a reconstrução. Encontre mais detalhes em [este artigo](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

### Step 4: Post-process and return result in desired classical format

Now we retrieve the result of each subexperiment run and reconstruct the expectation value of the uncut circuit:

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

### Passo 4: Pós-processar e retornar resultado no formato clássico desejado
Agora recuperamos o resultado de cada subexperimento executado e reconstruímos o valor esperado do circuito não cortado:

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Periodic boundary conditions with circuit cutting](/docs/tutorials/periodic-boundary-conditions-with-circuit-cutting)
- [Circuit cutting for depth reduction](/docs/tutorials/depth-reduction-with-circuit-cutting)
</Admonition>

## References


[1] Peng, T., Harrow, A. W., Ozols, M., & Wu, X. (2020). Simulating large quantum circuits on a small quantum computer. Physical review letters, 125(15), 150504.

[2] Tang, W., Tomesh, T., Suchara, M., Larson, J., & Martonosi, M. (2021, April). Cutqc: using small quantum computers for large quantum circuit evaluations. In Proceedings of the 26th ACM International conference on architectural support for programming languages and operating systems (pp. 473-486).

[3]  Perlin, M. A., Saleem, Z. H., Suchara, M., & Osborn, J. C. (2021). Quantum circuit cutting with maximum-likelihood tomography. npj Quantum Information, 7(1), 64.

[4]  Majumdar, R., & Wood, C. J. (2022). Error mitigated quantum circuit cutting. arXiv preprint arXiv:2211.13431.

[5]  Khare, T., Majumdar, R., Sangle, R., Ray, A., Seshadri, P. V., & Simmhan, Y. (2023). Parallelizing Quantum-Classical Workloads: Profiling the Impact of Splitting Techniques. In 2023 IEEE International Conference on Quantum Computing and Engineering (QCE) (Vol. 1, pp. 990-1000). IEEE.

[6]  Bhoumik, D., Majumdar, R., Saha, A., & Sur-Kolay, S. (2023). Distributed Scheduling of Quantum Circuits with Noise and Time Optimization. arXiv preprint arXiv:2309.06005.

[7]  Majumdar, R. (2024). Efficient Reduction of Resources and Noise in Discrete Quantum Computing Circuits (Doctoral dissertation, Indian Statistical Institute - Kolkata). https://www.proquest.com/openview/b481def90b1cc80e6b58a77c99e8385c/1?pq-origsite=gscholar&cbl=2026366&diss=y